In [1]:
import torch
import psutil
import os
import boto3
import tarfile
from transformers import AutoModelForCausalLM, AutoTokenizer

# Função para obter tamanho em disco
def get_model_size(model_path):
    if not os.path.exists(model_path):
        return 'N/A'
    total_size = 0
    for dirpath, _, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / 1024**2  # Em MB

# Configurações
model_base = 'unsloth/Llama-3.2-1B-Instruct'
s3_model_finetuned = 'nlp-genai-grupo5/output/grupo5-fine-tuning-pytorch-20250728-1614/output/model.tar.gz'
local_model_path = '/tmp/model-fine-tuned'
device = 'cuda'

# Baixar e descompactar modelo fine-tuned do S3
s3_client = boto3.client('s3', region_name='eu-west-1')
s3_client.download_file('sagemaker-eu-west-1-267567228900', s3_model_finetuned, '/tmp/model.tar.gz')
with tarfile.open('/tmp/model.tar.gz', 'r:gz') as tar:
    tar.extractall(local_model_path)

# Verificar o conteúdo do diretório descompactado
print("Conteúdo de", local_model_path, ":", os.listdir(local_model_path))

# Ajustar local_model_path para a pasta correta (merged_16bit ou merged_4bit)
possible_folders = [ 'merged_4bit']  #'merged_16bit']
finetuned_model_path = local_model_path
for folder in possible_folders:
    candidate_path = os.path.join(local_model_path, folder)
    if os.path.exists(candidate_path) and os.path.isdir(candidate_path):
        finetuned_model_path = candidate_path
        break
print(f"Usando caminho do modelo fine-tuned: {finetuned_model_path}")

# Medir RAM inicial
ram_before = psutil.virtual_memory().used / 1024**2  # Em MB

#  Modelo Base 
model = AutoModelForCausalLM.from_pretrained(model_base, load_in_4bit=True).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_base)
vram_base = torch.cuda.memory_allocated() / 1024**2  # Em MB
ram_after_base = psutil.virtual_memory().used / 1024**2  # Em MB
disk_base = 'N/A'
del model, tokenizer
torch.cuda.empty_cache()

#  Modelo Fine-Tuned (Completo) 
model = AutoModelForCausalLM.from_pretrained(finetuned_model_path, load_in_4bit=True).to(device)
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)
vram_finetuned = torch.cuda.memory_allocated() / 1024**2  # Em MB
ram_after_finetuned = psutil.virtual_memory().used / 1024**2  # Em MB
disk_finetuned = get_model_size(finetuned_model_path)
del model, tokenizer
torch.cuda.empty_cache()

# Exibir resultados
print('\n📊 Resultados de Memória Estática:')
print(f'Modelo Base - VRAM: {vram_base:.2f} MB, RAM: {ram_after_base - ram_before:.2f} MB, Disco: {disk_base} MB')
print(f'Modelo Fine-Tuned - VRAM: {vram_finetuned:.2f} MB, RAM: {ram_after_finetuned - ram_before:.2f} MB, Disco: {disk_finetuned:.2f} MB')

# Salvar resultados em arquivo
with open('relatorio_memoria_estatica_4bit.txt', 'w', encoding='utf-8') as f:
    f.write('📊 RELATÓRIO DE MEMÓRIA ESTÁTICA\n')
    f.write('='*50 + '\n')
    f.write(f'Modelo Base - VRAM: {vram_base:.2f} MB, RAM: {ram_after_base - ram_before:.2f} MB, Disco: {disk_base} MB\n')
    f.write(f'Modelo Fine-Tuned - VRAM: {vram_finetuned:.2f} MB, RAM: {ram_after_finetuned - ram_before:.2f} MB, Disco: {disk_finetuned:.2f} MB\n')

# Enviar para S3
s3_client.upload_file('relatorio_memoria_estatica_4bit.txt', 'sagemaker-eu-west-1-267567228900', 'nlp-genai-grupo5/output/relatorio_memoria_estatica.txt')
print('Arquivo relatorio_memoria_estatica.txt enviado para S3')

/tmp/ipykernel_3052/2367585433.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(local_model_path)


Conteúdo de /tmp/model-fine-tuned : ['merged_16bit', 'training_args.bin', 'README.md', 'checkpoint-405', 'adapter_model.safetensors', 'tokenizer.json', 'special_tokens_map.json', 'lora_adapter', 'runs', 'chat_template.jinja', 'tokenizer_config.json', 'merged_4bit', 'adapter_config.json']
Usando caminho do modelo fine-tuned: /tmp/model-fine-tuned/merged_4bit


2025-07-29 21:39:13.542882: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-29 21:39:13.557631: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753825153.576553    3052 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753825153.582262    3052 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-29 21:39:13.601491: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr


📊 Resultados de Memória Estática:
Modelo Base - VRAM: 1023.18 MB, RAM: 704.57 MB, Disco: N/A MB
Modelo Fine-Tuned - VRAM: 1574.37 MB, RAM: 713.69 MB, Disco: 1067.77 MB
Arquivo relatorio_memoria_estatica.txt enviado para S3
